# Skipstabilitet: fra ballastplan til rulling i bølger

## Pilotprosjekt for Matematikk 1

Et skip må ikke bare flyte. Det må også kunne rette seg opp etter vind, bølger, lastforskyvning eller andre påvirkninger. I dette prosjektet bygger vi en forenklet modell for skipets stabilitet og rullebevegelse.

Prosjektet har tre hoveddeler:

1. **Ballast og tyngdepunkt:** et lineært system for plassering av ballast
2. **Små rullebevegelser:** en lineær andreordens ODE som løses for hånd og med Euler
3. **Store vinkler og bølger:** et ikke-lineært vektor-ODE-system

### Læringsmål

Etter prosjektet skal du kunne

- bruke masse- og momentbalanser til å finne et samlet tyngdepunkt,
- sette opp og løse et lineært system for ballastmasser,
- beregne en forenklet metasenterhøyde,
- tolke positiv, null og negativ initial stabilitet,
- løse en lineær andreordens ODE med konstante koeffisienter,
- koble karakteristiske røtter til egenverdier i et $2\times2$-system,
- bruke Eulers metode på et vektor-ODE-system,
- sammenligne lineært og ikke-lineært rettende moment,
- undersøke rulling under regelmessig bølgepåvirkning,
- diskutere begrensningene i en pedagogisk stabilitetsmodell.

### Viktig avgrensning

Modellen er laget for undervisning. Den kan ikke brukes som dokumentasjon av sikkerhet, klassing eller forskriftsmessig stabilitet for et virkelig fartøy. Reelle stabilitetsberegninger krever detaljert skroggeometri, lastetilstander, fribord, vanntette grenser, fri væskeoverflate, skadeforhold og gjeldende maritime krav.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fire grunnbegreper

## Tyngdepunkt $G$

Skipets samlede vekt kan regnes å virke nedover gjennom tyngdepunktet $G$. Last plassert høyt hever $G$, mens ballast plassert lavt senker $G$.

## Oppdriftssenter $B$

Oppdriften virker oppover gjennom tyngdepunktet til det fortrengte vannvolumet. Dette punktet kalles oppdriftssenteret $B$.

## Metasenter $M$

Ved små krengningsvinkler brukes metasenteret $M$ til å beskrive initial stabilitet. Avstanden

$$GM=KM-KG$$

kalles metasenterhøyden.

- $GM>0$: den oppreiste stillingen er initialt stabil
- $GM=0$: modellen har ingen lineær rettende virkning
- $GM<0$: den oppreiste stillingen er ustabil

## Rettende arm $GZ$

Når skipet krenger, virker tyngden og oppdriften langs ulike vertikale linjer. Den horisontale avstanden mellom linjene kalles den rettende armen $GZ$.

Det rettende momentet er

$$M_R=W\,GZ(\phi),$$

hvor $W=mg$ og $\phi$ er krengningsvinkelen.

For små vinkler brukes

$$GZ(\phi)\approx GM\sin\phi\approx GM\phi.$$

# Del A: Ballast, tyngdepunkt og initial stabilitet

## A.1 Samlet tyngdepunkt

La masse $m_j$ ha koordinater $(x_j,y_j,z_j)$. Skipets samlede masse er

$$M=\sum_jm_j.$$

Tyngdepunktet er

$$
x_G=\frac{\sum_jm_jx_j}{M},\qquad
y_G=\frac{\sum_jm_jy_j}{M},\qquad
z_G=\frac{\sum_jm_jz_j}{M}.
$$

Høyden $z_G$ over kjølen betegnes ofte $KG$.

## Oppgave A1: Beregn tyngdepunktet

Et forenklet fartøy består av skrog, maskineri, dekksutstyr og last. Koordinatene er gitt i meter relativt til et valgt koordinatsystem i skipet.

Beregn samlet masse og tyngdepunkt.

In [ ]:
komponenter = [
    # navn, masse i tonn, x, y, z i meter
    ("skrog",       520.0, 30.0,  0.0, 3.2),
    ("maskineri",   140.0, 18.0, -0.5, 2.0),
    ("dekksutstyr",  45.0, 35.0,  0.8, 7.5),
    ("last",        260.0, 33.0,  0.6, 5.8)
]

m = np.array([rad[1] for rad in komponenter])
r = np.array([[rad[2], rad[3], rad[4]] for rad in komponenter])

M = ...
r_G = ...

print("Samlet masse:", M, "tonn")
print("Tyngdepunkt [xG, yG, KG]:", r_G, "m")

## A.2 Tre ballasttanker

Vi ønsker å fylle tre ballasttanker med ukjente vannmasser $m_1,m_2,m_3$.

Kravene er:

1. total ballastmasse skal være $M_B$,
2. ballastens samlede vertikale tyngdepunkt skal være $z_B$,
3. ballastens samlede tverrskips tyngdepunkt skal være $y_B=0$.

Dette gir

$$
\begin{aligned}
m_1+m_2+m_3&=M_B,\\
z_1m_1+z_2m_2+z_3m_3&=M_Bz_B,\\
y_1m_1+y_2m_2+y_3m_3&=0.
\end{aligned}
$$

## Oppgave A2: Lineært system for ballast

Tankene har posisjonene

$$
(y_1,z_1)=(-5.0,1.0),\quad
(y_2,z_2)=(0.0,2.0),\quad
(y_3,z_3)=(4.0,0.8).
$$

Vi ønsker

$$M_B=180\ \text{tonn},\qquad z_B=1.25\ \text{m}.$$

Sett opp $Ax=b$ og løs systemet. Kontroller residualet og at ingen tank får negativ masse.

In [ ]:
y_tank = np.array([-5.0, 0.0, 4.0])
z_tank = np.array([1.0, 2.0, 0.8])

M_B = 180.0
z_B = 1.25

A_ballast = np.array([
    [1.0, 1.0, 1.0],
    [z_tank[0], z_tank[1], z_tank[2]],
    [y_tank[0], y_tank[1], y_tank[2]]
])

b_ballast = np.array([M_B, M_B*z_B, 0.0])

m_ballast = ...
print("Ballastmasser:", m_ballast, "tonn")
print("Residual:", ...)
print("Alle masser ikke-negative:", ...)

## Oppgave A3: Tankkapasitet og realisme

Anta at tankkapasitetene er

$$[80,100,90]\ \text{tonn}.$$

Kontroller om ballastplanen er fysisk gjennomførbar. Dersom en kapasitet overskrides, forklar hvorfor løsningen av $Ax=b$ likevel er matematisk korrekt, men teknisk ubrukelig.

Endre $z_B$ og undersøk når systemet begynner å gi negative ballastmasser.

In [ ]:
kapasitet = np.array([80.0, 100.0, 90.0])

print("Innenfor kapasitet:", ...)

# Lag gjerne en parameterstudie av ønsket z_B.

## A.3 Oppdatert tyngdepunkt

Ballasttankene antas å ligge ved samme langskipsposisjon $x=25$ m. Kombiner ballast og opprinnelige komponenter og beregn nytt tyngdepunkt.

In [ ]:
x_ballast = 25.0
r_ballast = np.column_stack([
    np.full(3, x_ballast),
    y_tank,
    z_tank
])

m_alle = np.concatenate([m, m_ballast])
r_alle = np.vstack([r, r_ballast])

M_ny = ...
r_G_ny = ...

print("Ny totalmasse:", M_ny, "tonn")
print("Nytt tyngdepunkt:", r_G_ny, "m")

## A.4 En enkel lektermodell for metasenterhøyde

For et rektangulært skrog med lengde $L$, bredde $b$ og dypgang $T$ bruker vi

$$\nabla=LbT$$

for fortrengt volum. Vannlinjens andre arealmoment om lengdeaksen er

$$I_{WP}=\frac{Lb^3}{12}.$$

Dermed

$$BM=\frac{I_{WP}}{\nabla}=\frac{b^2}{12T}.$$

Vi bruker også tilnærmingen

$$KB\approx\frac{T}{2}.$$

Da blir

$$
\boxed{GM=\frac{T}{2}+\frac{b^2}{12T}-KG.}
$$

## Oppgave A4: Beregn $GM$

Bruk

$$L=60\ \text{m},\qquad b=12\ \text{m},\qquad T=4.0\ \text{m}.$$

Beregn $KB$, $BM$, $KM$ og $GM$ før og etter ballast. Kommenter virkningen av ballastplasseringen.

In [ ]:
L_skip = 60.0
bredde = 12.0
dypgang = 4.0

KB = ...
I_WP = ...
volum = ...
BM = ...
KM = ...

KG_før = r_G[2]
KG_etter = r_G_ny[2]

GM_før = ...
GM_etter = ...

print("KB, BM, KM:", KB, BM, KM)
print("GM før ballast:", GM_før, "m")
print("GM etter ballast:", GM_etter, "m")

## Oppgave A5: Bredde, dypgang og høyt plassert last

Undersøk hvordan $GM$ endres når

1. bredden økes,
2. dypgangen økes,
3. en last flyttes høyere,
4. ballast flyttes nærmere kjølen.

Forklar hvorfor en stor positiv $GM$ gir sterk initial stabilitet, men ikke nødvendigvis best komfort.

In [ ]:
bredder = np.linspace(8.0, 18.0, 100)
GM_bredde = ...

# Lag plott av GM som funksjon av bredde.

# Del B: Små rullebevegelser

La $\phi(t)$ være krengningsvinkelen i radianer. Rotasjonsbalansen er

$$I_\phi\ddot\phi=\sum M.$$

Vi bruker

- dempingsmoment $-c_\phi\dot\phi$,
- linearisert rettende moment $-WGM\phi$.

Dermed får vi

$$
\boxed{
I_\phi\ddot\phi+c_\phi\dot\phi+WGM\phi=0.
}
$$

Her er $I_\phi$ et effektivt treghetsmoment for rulling. Det kan omfatte både skipets egen treghet og en forenklet virkning av omkringliggende vann.

## Oppgave B1: Standardform

Del ligningen på $I_\phi$ og skriv den som

$$
\ddot\phi+2\zeta\omega_n\dot\phi+\omega_n^2\phi=0.
$$

Vis at

$$
\omega_n=\sqrt{\frac{WGM}{I_\phi}},
$$

og

$$
2\zeta\omega_n=\frac{c_\phi}{I_\phi}.
$$

Hva skjer med $\omega_n$ dersom $GM$ øker?

## Oppgave B2: Karakteristisk polynom

Det karakteristiske polynomet er

$$
r^2+\frac{c_\phi}{I_\phi}r+\frac{WGM}{I_\phi}=0.
$$

Klassifiser løsningen som underdempet, kritisk dempet eller overdempet. Forklar hva negativ $GM$ gjør med likevekten $\phi=0$.

## Oppgave B3: Et underdempet skip

Bruk skalerte størrelser

$$I_\phi=1.8\cdot10^8\ \mathrm{kg\,m^2},$$

$$M=1.15\cdot10^6\ \mathrm{kg},\qquad g=9.81\ \mathrm{m/s^2},$$

$$GM=1.20\ \mathrm m,\qquad c_\phi=1.2\cdot10^7\ \mathrm{N\,m\,s}.$$

Begynnelsesbetingelsene er

$$\phi(0)=8^\circ,\qquad \dot\phi(0)=0.$$

Finn den analytiske løsningen for hånd og beregn naturlig periode og dempningsforhold.

In [ ]:
I_phi = 1.8e8
M_kg = 1.15e6
g = 9.81
W = M_kg*g
GM = 1.20
c_phi = 1.2e7

phi0 = np.deg2rad(8.0)
omega0 = 0.0

omega_n = ...
zeta = ...

print("Naturlig vinkelfrekvens:", omega_n, "rad/s")
print("Naturlig periode:", 2*np.pi/omega_n, "s")
print("Dempningsforhold:", zeta)

## Oppgave B4: Skriv inn håndløsningen

For det underdempede tilfellet er

$$
\phi(t)=e^{-\zeta\omega_nt}
\left(C_1\cos\omega_dt+C_2\sin\omega_dt\right),
$$

hvor

$$\omega_d=\omega_n\sqrt{1-\zeta^2}.$$

Bestem $C_1,C_2$ fra begynnelsesbetingelsene og plott vinkelen i grader.

In [ ]:
t = np.linspace(0.0, 60.0, 2000)
omega_d = ...
C1 = ...
C2 = ...
phi_eksakt = ...

plt.plot(t, np.rad2deg(phi_eksakt))
plt.axhline(0.0, color="black", linewidth=1)
plt.xlabel("Tid s")
plt.ylabel("Krengningsvinkel grader")
plt.grid()
plt.show()

## Oppgave B5: Førsteordenssystem og Euler

Sett

$$\omega=\dot\phi.$$

Da blir modellen

$$
\begin{aligned}
\dot\phi&=\omega,\\
\dot\omega&=-\frac{WGM}{I_\phi}\phi-\frac{c_\phi}{I_\phi}\omega.
\end{aligned}
$$

Implementer Euler og sammenlign med håndløsningen. Prøv flere steglengder.

In [ ]:
def lineær_rulling(t, x):
    phi, omega = x
    dphi = ...
    domega = ...
    return np.array([dphi, domega])


def euler_system(f, x0, T, h):
    n = int(round(T/h))
    t = np.linspace(0.0, n*h, n + 1)
    X = np.zeros((n + 1, len(x0)))
    X[0] = x0

    for k in range(n):
        X[k + 1] = ...

    return t, X


t_B, X_B = euler_system(
    lineær_rulling,
    x0=np.array([phi0, omega0]),
    T=60.0,
    h=0.02
)

# Sammenlign med eksakt løsning.

## Oppgave B6: Egenverdier

Systemmatrisen er

$$
A=
\begin{pmatrix}
0&1\\
-WGM/I_\phi&-c_\phi/I_\phi
\end{pmatrix}.
$$

Beregn egenverdiene. Kontroller at de er de samme som røttene i det karakteristiske polynomet. Dersom matrisen kan diagonaliseres, skriv den som

$$A=PDP^{-1}.$$

In [ ]:
A_roll = np.array([
    [0.0, 1.0],
    [-W*GM/I_phi, -c_phi/I_phi]
])

egenverdier, P = ...
print("Egenverdier:", egenverdier)
print("Egenvektorer:
", P)

# Kontroller diagonaliserbarhet og A = P D P^{-1}.

## Oppgave B7: Ballast og rulleperiode

Bruk flere positive $GM$-verdier og sammenlign

- naturlig periode,
- maksimal vinkel etter slipp,
- hvor raskt bevegelsen dempes.

Diskuter begrepene

- **stivt skip:** stor $GM$, raskere rulling,
- **mykt eller tender skip:** liten positiv $GM$, langsommere rulling.

# Del C: Ikke-lineær stabilitet og bølger

Ved større vinkler er tilnærmingen

$$GZ\approx GM\phi$$

for grov. Vi går derfor tilbake til momentbalansen

$$
I_\phi\ddot\phi+c_\phi\dot\phi+WGZ(\phi)
=M_{bølge}(t)+M_H.
$$

Her er

- $M_{bølge}(t)$ et tidsavhengig bølgemoment,
- $M_H$ et konstant krenge­moment, for eksempel fra tverrskips lastforskyvning.

Som første ikke-lineære modell bruker vi

$$GZ(\phi)=GM\sin\phi.$$

## Oppgave C1: Vektor-ODE

Vis at modellen kan skrives

$$
\boxed{
\begin{aligned}
\dot\phi&=\omega,\\
\dot\omega&=
\frac{-c_\phi\omega-WGZ(\phi)+M_{bølge}(t)+M_H}{I_\phi}.
\end{aligned}}
$$

Implementer modellen med $M_{bølge}=M_H=0$ og sammenlign $GZ=GM\sin\phi$ med den lineære modellen for startvinklene

$$5^\circ,\quad20^\circ,\quad45^\circ.$$

In [ ]:
def GZ_sinus(phi):
    return GM*np.sin(phi)


def ikke_lineær_fri(t, x):
    phi, omega = x
    dphi = omega
    domega = ...
    return np.array([dphi, domega])

# Simuler flere startvinkler og sammenlign.

## C.2 En pedagogisk stabilitetskurve

En virkelig $GZ$-kurve kan stige, nå et maksimum og senere falle til null ved en vinkel der den rettende armen forsvinner.

Vi bruker modellen

$$
\boxed{
GZ(\phi)=GM\sin\phi
\left(1-\frac{\phi^2}{\phi_v^2}\right),
}
$$

for vinkler i modellområdet. Her er $\phi_v$ en valgt vinkel for vanishing stability.

Modellen har

$$GZ(0)=0,\qquad GZ'(0)=GM,\qquad GZ(\phi_v)=0.$$

Dette er en pedagogisk kurve, ikke en faktisk skrogspesifikk stabilitetskurve.

## Oppgave C2: Plott $GZ$-kurven

Bruk

$$\phi_v=75^\circ.$$

Plott den lineære tilnærmingen, sinusmodellen og den pedagogiske $GZ$-kurven mellom $-90^\circ$ og $90^\circ$.

Finn numerisk

- vinkelen for maksimal positiv rettende arm,
- maksimal $GZ$,
- nullpunktene.

In [ ]:
phi_v = np.deg2rad(75.0)


def GZ_pedagogisk(phi):
    return GM*np.sin(phi)*(1 - (phi/phi_v)**2)


phi_grid = np.deg2rad(np.linspace(-90.0, 90.0, 1001))

# Plott tre modeller og finn maksimum/nullpunkter.

## Oppgave C3: Dynamikk med begrenset stabilitetsrange

Bruk den pedagogiske $GZ$-kurven i ODE-systemet. Simuler fri rulling fra flere startvinkler.

Stopp simuleringen dersom

$$|\phi|\ge90^\circ.$$

Dette er et numerisk og pedagogisk stoppkriterium, ikke en universell kapseisingsgrense.

In [ ]:
def euler_med_stopp(f, x0, T, h, phi_stopp):
    n = int(round(T/h))
    t = np.linspace(0.0, n*h, n + 1)
    X = np.zeros((n + 1, len(x0)))
    X[0] = x0
    siste = n

    for k in range(n):
        X[k + 1] = ...
        if abs(X[k + 1, 0]) >= phi_stopp or not np.all(np.isfinite(X[k + 1])):
            siste = k + 1
            break

    return t[:siste + 1], X[:siste + 1]


def rulling_GZ(t, x):
    phi, omega = x
    return np.array([
        omega,
        ...
    ])

# Simuler passende startvinkler.

## C.3 Regelmessige bølger

Vi modellerer bølgemomentet som

$$
M_{bølge}(t)=M_0\sin(\Omega t).
$$

Dette representerer en regelmessig påvirkning fra siden. Modellen beskriver ikke en full sjøtilstand.

## Oppgave C4: Frekvensstudie

Bruk en moderat momentamplitude og undersøk

1. $\Omega=0.5\omega_n$,
2. $\Omega\approx\omega_n$,
3. $\Omega=2\omega_n$.

Sammenlign maksimal krengningsvinkel etter at den første transienten har avtatt. Forklar hvorfor responsen kan bli stor nær den naturlige rullefrekvensen.

In [ ]:
M0 = 2.0e6


def lag_bølgemodell(Omega, M_H=0.0):
    def modell(t, x):
        phi, omega = x
        M_bølge = M0*np.sin(Omega*t)
        dphi = omega
        domega = ...
        return np.array([dphi, domega])
    return modell

# Simuler de tre frekvensene og sammenlign.

## Oppgave C5: Flyttet last og ny likevekt

Hvis en masse forskyves tverrskips, flyttes tyngdepunktet. En enkel modell for et konstant krenge­moment er

$$M_H=W y_G.$$

For liten vinkel og uten bølger oppfyller likevekten omtrent

$$WGM\phi^*=Wy_G,
$$

slik at

$$
\boxed{\phi^*\approx\frac{y_G}{GM}.}
$$

Simuler modellen med et lite konstant $M_H$ og sammenlign den numeriske likevekten med småvinkelanslaget.

In [ ]:
y_G_forskyvning = 0.10
M_H = W*y_G_forskyvning
phi_likevekt_liten = y_G_forskyvning/GM

print("Småvinkelanslag i grader:", np.rad2deg(phi_likevekt_liten))

# Simuler uten bølger og finn numerisk likevektsvinkel.

## Oppgave C6: Ballast som designvalg

Velg tre ballastplaner som gir ulike positive $GM$-verdier. For hver plan, beregn eller simuler

- naturlig rulleperiode,
- respons nær resonans,
- maksimal vinkel under den valgte bølgepåvirkningen,
- likevektsvinkel ved en lastforskyvning.

Diskuter kompromisset mellom

- initial stabilitet,
- rullekomfort,
- dynamisk respons,
- praktiske tankkapasiteter.

# Modellkritikk og videreføring

## Oppgave D1: Begrensninger

Diskuter minst fem punkter:

- Skroget er representert med en rektangulær lektermodell.
- $KB\approx T/2$ gjelder bare den valgte forenklingen.
- $GM$ beskriver bare initial stabilitet ved små vinkler.
- Dempingen er lineær i rullehastigheten.
- Det effektive treghetsmomentet er gitt som en konstant.
- Bølgemomentet er sinusformet og én-dimensjonalt.
- Bare rulling er med, ikke stamping, hiv, giring eller slingring.
- Lasten antas fast, med unntak av et gitt krenge­moment.
- Fri væskeoverflate i delvis fylte tanker er utelatt.
- Den pedagogiske $GZ$-kurven kommer ikke fra faktisk skroggeometri.
- Vann på dekk, åpninger og skade er utelatt.

## Videreføring

- bruk av tabulerte eller beregnede $GZ$-data,
- interpolasjon mellom stabilitetsdata,
- ikke-lineær rulledemping,
- tilfeldig bølgepåvirkning,
- kobling mellom rulling og stamping,
- fri overflate i tanker,
- skade- og fyllingsscenarier,
- estimering av demping fra et rulleforsøk.

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvordan masse- og momentbalanser bestemmer tyngdepunktet,
2. hvordan ballastproblemet ble skrevet som $Ax=b$,
3. hvordan ballast påvirket $KG$ og $GM$,
4. hvordan den lineære rulleligningen ble utledet,
5. hva karakteristiske røtter og egenverdier forteller om stabilitet,
6. hvordan Euler-løsningen sammenlignet med håndløsningen,
7. når småvinkelmodellen begynte å avvike,
8. hvordan $GZ$-kurven bestemte det rettende momentet,
9. hvordan bølgefrekvens og ballast påvirket rullingen,
10. hvorfor modellen ikke kan brukes som sikkerhetsdokumentasjon for et virkelig skip.

## Referanser for videre lesning

- Innførende materiale om metasenterhøyde, rettende arm og statisk stabilitet.
- ITTCs retningslinjer for estimering av rulledemping.

Studentene trenger ikke lese eksterne kilder for å gjennomføre prosjektet.